# 🎬 Netflix Content Analysis — Exploratory Data Analysis

> **Dataset:** Netflix Movies and TV Shows  
> **Records:** 8,807 titles | **Features:** 12 columns  
> **Goal:** Understand content trends, patterns, and distribution on Netflix

---

## 📋 Table of Contents
1. [Setup & Imports](#1)
2. [Load & Preview Data](#2)
3. [Data Cleaning](#3)
4. [Univariate Analysis — Single Variable Exploration](#4)
5. [Bivariate Analysis — Two Variable Relationships](#5)
6. [Time Series — Content Growth Over Years](#6)
7. [Genre Deep Dive](#7)
8. [Country-wise Analysis](#8)
9. [Word Cloud — Descriptions](#9)
10. [Key Insights & Conclusions](#10)


---
## 1. Setup & Imports <a id='1'></a>

We import all the libraries we need. Think of libraries as toolboxes:
- **pandas** → working with table data (like Excel in Python)
- **numpy** → math and number operations
- **matplotlib / seaborn** → creating charts and graphs
- **plotly** → interactive charts
- **wordcloud** → text visualization
- **warnings** → suppress unimportant messages

In [1]:
# ═══════════════════════════════════════════════════════════════════
# IMPORTANT: Run this cell FIRST before any other cell!
# Shortcut: Kernel → Restart & Run All  (runs everything at once)
# ═══════════════════════════════════════════════════════════════════

# Standard libraries
import os
import warnings
warnings.filterwarnings('ignore')

import pandas as pd          # Data manipulation (like Excel in Python)
import numpy as np           # Numerical operations

# Visualization libraries
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud, STOPWORDS

# Create plots directory so savefig never fails
os.makedirs('plots', exist_ok=True)

# ── Global style settings ──────────────────────────────────────────────────
sns.set_theme(style='darkgrid', palette='Set2')
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.titlesize': 15,
    'axes.labelsize': 12,
})

# Brand colours — used throughout the notebook
NETFLIX_RED  = '#E50914'
NETFLIX_DARK = '#141414'

print('✅ All libraries loaded!')
print('   pandas  :', pd.__version__)
print('   seaborn :', sns.__version__)
print('   matplotlib:', matplotlib.__version__)


✅ All libraries loaded!
   pandas  : 3.0.2
   seaborn : 0.13.2
   matplotlib: 3.10.8


---
## 2. Load & Preview Data <a id='2'></a>

We load the CSV file into a **DataFrame** — think of it as a Python version of a spreadsheet table.

In [ ]:
# Load the dataset
df = pd.read_csv('netflix_titles.csv')

print(f'📊 Dataset Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'   → This means we have {df.shape[0]:,} Netflix titles and {df.shape[1]} attributes for each title.')
print()

# Display first 5 rows
df.head()

In [ ]:
# Column names and their data types
# dtype = data type: object means text, int64 means whole numbers
print('📝 Column Info:')
df.info()

In [ ]:
# Check for missing values — NaN means "Not a Number" / empty cell
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
}).sort_values('Missing %', ascending=False)

print('❓ Missing Values Summary:')
print(missing_df[missing_df['Missing Count'] > 0])

In [ ]:
# ── Guard: ensure imports ran even if you skipped cell 1 ─────────────────
try:
    sns
except NameError:
    import seaborn as sns
    import matplotlib.pyplot as plt
    sns.set_theme(style='darkgrid')
    NETFLIX_RED = '#E50914'
    print('⚠️  Imports were missing — loaded them now.')
    print('   Tip: Use Kernel → Restart & Run All for a clean run.')

# ── Visualise missing values as a heatmap ─────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 3))

# isnull() returns True (=1) where data is missing
# We sample 500 rows so the chart is not too large
sns.heatmap(
    df.sample(500, random_state=42).isnull(),
    cbar=False, yticklabels=False,
    cmap=['#2ecc71', NETFLIX_RED],   # green = present, red = missing
    ax=ax
)
ax.set_title('Missing Value Map (500 random rows) — Red = Missing', pad=12)
plt.tight_layout()
plt.savefig('plots/01_missing_values.png', bbox_inches='tight')
plt.show()


---
## 3. Data Cleaning <a id='3'></a>

Real-world data is messy. We fix:
- **Missing values** → fill with 'Unknown' or drop rows
- **Date columns** → convert text dates to proper datetime objects
- **Duration** → split into number + unit (minutes / seasons)

In [ ]:
import os
os.makedirs('plots', exist_ok=True)  # create folder to save charts

# ── Fill missing text fields with 'Unknown' ──────────────────────────────────
# fillna() replaces NaN with whatever value you give it
df['director'].fillna('Unknown', inplace=True)
df['cast'].fillna('Unknown', inplace=True)
df['country'].fillna('Unknown', inplace=True)
df['rating'].fillna('Unknown', inplace=True)

# ── Drop rows where date_added or duration is missing ────────────────────────
# Only 10 + 3 rows affected — safe to drop
df.dropna(subset=['date_added', 'duration'], inplace=True)

# ── Convert date_added to datetime ───────────────────────────────────────────
# to_datetime() parses text like 'September 25, 2021' into a real date object
df['date_added'] = pd.to_datetime(df['date_added'].str.strip())
df['year_added'] = df['date_added'].dt.year    # extract year
df['month_added'] = df['date_added'].dt.month  # extract month (1-12)
df['month_name'] = df['date_added'].dt.strftime('%b')  # 'Jan', 'Feb', ...

# ── Parse duration ────────────────────────────────────────────────────────────
# Movies: '90 min'  →  duration_value=90,  duration_unit='min'
# TV Shows: '3 Seasons' → duration_value=3, duration_unit='Seasons'
df['duration_value'] = df['duration'].str.extract(r'(\d+)').astype(int)
df['duration_unit']  = df['duration'].str.extract(r'(\D+)').str.strip()

print('✅ Cleaning complete!')
print(f'   Remaining rows: {len(df):,}')
df[['title', 'type', 'date_added', 'year_added', 'duration', 'duration_value', 'duration_unit']].head()

---
## 4. Univariate Analysis — Exploring One Variable at a Time <a id='4'></a>

**Univariate** = one variable. We look at each column individually to understand its distribution.

In [ ]:
# ── Safety: re-import if kernel was restarted mid-notebook ───────────────
try:
    sns, plt, pd, NETFLIX_RED
except NameError:
    import pandas as pd, seaborn as sns, matplotlib.pyplot as plt
    from wordcloud import WordCloud, STOPWORDS
    import os, warnings; warnings.filterwarnings('ignore')
    os.makedirs('plots', exist_ok=True)
    sns.set_theme(style='darkgrid')
    plt.rcParams.update({'figure.dpi':120,'axes.titlesize':15,'axes.labelsize':12})
    NETFLIX_RED='#E50914'; NETFLIX_DARK='#141414'
    print('⚠️  Re-imported libraries. Run Kernel → Restart & Run All for best results.')

# ─────────────────────────────────────────────────────────────────────────────
# 4.1  Movies vs TV Shows — Pie + Bar
# ─────────────────────────────────────────────────────────────────────────────
type_counts = df['type'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Content Type Distribution on Netflix', fontsize=16, fontweight='bold')

# Pie chart
axes[0].pie(
    type_counts,
    labels=type_counts.index,
    autopct='%1.1f%%',         # show percentages
    colors=[NETFLIX_RED, '#221f1f'],
    startangle=90,
    explode=[0.05, 0],         # slightly separate the first slice
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
axes[0].set_title('Proportion')

# Bar chart — value_counts already sorted
sns.barplot(x=type_counts.index, y=type_counts.values, ax=axes[1],
            palette=[NETFLIX_RED, '#555555'])
for p in axes[1].patches:  # add count labels on bars
    axes[1].annotate(f'{int(p.get_height()):,}',
                     (p.get_x() + p.get_width() / 2, p.get_height() + 50),
                     ha='center', fontweight='bold')
axes[1].set_title('Count')
axes[1].set_ylabel('Number of Titles')

plt.tight_layout()
plt.savefig('plots/02_content_type.png', bbox_inches='tight')
plt.show()

print(f'\n🎯 Insight: Movies dominate Netflix at {type_counts["Movie"]/len(df)*100:.1f}% of all content.')

In [ ]:
# ── Safety: re-import if kernel was restarted mid-notebook ───────────────
try:
    sns, plt, pd, NETFLIX_RED
except NameError:
    import pandas as pd, seaborn as sns, matplotlib.pyplot as plt
    from wordcloud import WordCloud, STOPWORDS
    import os, warnings; warnings.filterwarnings('ignore')
    os.makedirs('plots', exist_ok=True)
    sns.set_theme(style='darkgrid')
    plt.rcParams.update({'figure.dpi':120,'axes.titlesize':15,'axes.labelsize':12})
    NETFLIX_RED='#E50914'; NETFLIX_DARK='#141414'
    print('⚠️  Re-imported libraries. Run Kernel → Restart & Run All for best results.')

# ─────────────────────────────────────────────────────────────────────────────
# 4.2  Content Ratings Distribution
# ─────────────────────────────────────────────────────────────────────────────
# Rating = age/audience classification (TV-MA=Mature, TV-14=14+, etc.)
rating_counts = df[df['rating'] != 'Unknown']['rating'].value_counts()

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(rating_counts.index, rating_counts.values,
              color=[NETFLIX_RED if i < 3 else '#888' for i in range(len(rating_counts))],
              edgecolor='white', linewidth=0.8)

# Annotate each bar with count
for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, h + 20, str(int(h)),
            ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_title('Content Rating Distribution', pad=12)
ax.set_xlabel('Rating')
ax.set_ylabel('Number of Titles')

plt.tight_layout()
plt.savefig('plots/03_ratings.png', bbox_inches='tight')
plt.show()

print('\n📌 Rating Guide: TV-MA = Mature Audiences | TV-14 = 14+ | R = Restricted | PG-13 = 13+')

In [ ]:
# ── Safety: re-import if kernel was restarted mid-notebook ───────────────
try:
    sns, plt, pd, NETFLIX_RED
except NameError:
    import pandas as pd, seaborn as sns, matplotlib.pyplot as plt
    from wordcloud import WordCloud, STOPWORDS
    import os, warnings; warnings.filterwarnings('ignore')
    os.makedirs('plots', exist_ok=True)
    sns.set_theme(style='darkgrid')
    plt.rcParams.update({'figure.dpi':120,'axes.titlesize':15,'axes.labelsize':12})
    NETFLIX_RED='#E50914'; NETFLIX_DARK='#141414'
    print('⚠️  Re-imported libraries. Run Kernel → Restart & Run All for best results.')

# ─────────────────────────────────────────────────────────────────────────────
# 4.3  Movie Duration Distribution (Histogram)
# ─────────────────────────────────────────────────────────────────────────────
# A histogram groups values into 'bins' and shows how many fall in each bin
movies = df[df['type'] == 'Movie'].copy()

fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(movies['duration_value'], bins=40, color=NETFLIX_RED, edgecolor='white', alpha=0.85)

# Vertical lines for mean and median
mean_dur = movies['duration_value'].mean()
med_dur  = movies['duration_value'].median()
ax.axvline(mean_dur, color='yellow', linestyle='--', linewidth=2, label=f'Mean: {mean_dur:.0f} min')
ax.axvline(med_dur,  color='cyan',   linestyle='--', linewidth=2, label=f'Median: {med_dur:.0f} min')

ax.set_title('Distribution of Movie Durations (in minutes)')
ax.set_xlabel('Duration (minutes)')
ax.set_ylabel('Number of Movies')
ax.legend()

plt.tight_layout()
plt.savefig('plots/04_movie_duration.png', bbox_inches='tight')
plt.show()

print(f'\n🎯 Average movie length: {mean_dur:.0f} minutes  |  Median: {med_dur:.0f} minutes')
print(f'   Shortest: {movies["duration_value"].min()} min  |  Longest: {movies["duration_value"].max()} min')

In [ ]:
# ── Safety: re-import if kernel was restarted mid-notebook ───────────────
try:
    sns, plt, pd, NETFLIX_RED
except NameError:
    import pandas as pd, seaborn as sns, matplotlib.pyplot as plt
    from wordcloud import WordCloud, STOPWORDS
    import os, warnings; warnings.filterwarnings('ignore')
    os.makedirs('plots', exist_ok=True)
    sns.set_theme(style='darkgrid')
    plt.rcParams.update({'figure.dpi':120,'axes.titlesize':15,'axes.labelsize':12})
    NETFLIX_RED='#E50914'; NETFLIX_DARK='#141414'
    print('⚠️  Re-imported libraries. Run Kernel → Restart & Run All for best results.')

# ─────────────────────────────────────────────────────────────────────────────
# 4.4  TV Show Seasons Distribution
# ─────────────────────────────────────────────────────────────────────────────
shows = df[df['type'] == 'TV Show'].copy()
season_counts = shows['duration_value'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(season_counts.index, season_counts.values, color='#221f1f', edgecolor=NETFLIX_RED, linewidth=1.2)
ax.set_title('Number of Seasons per TV Show')
ax.set_xlabel('Number of Seasons')
ax.set_ylabel('Number of Shows')
ax.set_xticks(season_counts.index)

plt.tight_layout()
plt.savefig('plots/05_tv_seasons.png', bbox_inches='tight')
plt.show()

print(f'\n🎯 {shows[shows["duration_value"]==1].shape[0]/len(shows)*100:.1f}% of TV shows have only 1 season.')
print(f'   Max seasons: {shows["duration_value"].max()} seasons')

---
## 5. Bivariate Analysis — Two Variables Together <a id='5'></a>

**Bivariate** = two variables. We compare one thing against another to find relationships.

In [ ]:
# ── Safety: re-import if kernel was restarted mid-notebook ───────────────
try:
    sns, plt, pd, NETFLIX_RED
except NameError:
    import pandas as pd, seaborn as sns, matplotlib.pyplot as plt
    from wordcloud import WordCloud, STOPWORDS
    import os, warnings; warnings.filterwarnings('ignore')
    os.makedirs('plots', exist_ok=True)
    sns.set_theme(style='darkgrid')
    plt.rcParams.update({'figure.dpi':120,'axes.titlesize':15,'axes.labelsize':12})
    NETFLIX_RED='#E50914'; NETFLIX_DARK='#141414'
    print('⚠️  Re-imported libraries. Run Kernel → Restart & Run All for best results.')

# ─────────────────────────────────────────────────────────────────────────────
# 5.1  Rating Distribution split by Movie vs TV Show
# ─────────────────────────────────────────────────────────────────────────────
# crosstab = cross tabulation: counts of combinations of two columns
rating_type = pd.crosstab(df['rating'], df['type'])
rating_type = rating_type[rating_type.sum(axis=1) > 30]  # filter rare ratings

rating_type.plot(kind='bar', figsize=(12, 5), color=[NETFLIX_RED, '#555'],
                 edgecolor='white', linewidth=0.8)
plt.title('Ratings by Content Type (Movies vs TV Shows)')
plt.xlabel('Rating')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.legend(title='Type')
plt.tight_layout()
plt.savefig('plots/06_rating_by_type.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Safety: re-import if kernel was restarted mid-notebook ───────────────
try:
    sns, plt, pd, NETFLIX_RED
except NameError:
    import pandas as pd, seaborn as sns, matplotlib.pyplot as plt
    from wordcloud import WordCloud, STOPWORDS
    import os, warnings; warnings.filterwarnings('ignore')
    os.makedirs('plots', exist_ok=True)
    sns.set_theme(style='darkgrid')
    plt.rcParams.update({'figure.dpi':120,'axes.titlesize':15,'axes.labelsize':12})
    NETFLIX_RED='#E50914'; NETFLIX_DARK='#141414'
    print('⚠️  Re-imported libraries. Run Kernel → Restart & Run All for best results.')

# ─────────────────────────────────────────────────────────────────────────────
# 5.2  Movie Duration by Rating — Box Plot
# A box plot shows: min, Q1 (25%), median (50%), Q3 (75%), max
# ─────────────────────────────────────────────────────────────────────────────
top_ratings = df['rating'].value_counts().head(8).index
filtered = movies[movies['rating'].isin(top_ratings)]

fig, ax = plt.subplots(figsize=(13, 6))
sns.boxplot(data=filtered, x='rating', y='duration_value',
            order=top_ratings, palette='Reds_r', ax=ax)
ax.set_title('Movie Duration by Content Rating')
ax.set_xlabel('Rating')
ax.set_ylabel('Duration (minutes)')

# Add a note explaining box plots
ax.text(0.01, 0.97,
        'Box: 25th–75th percentile | Line inside: Median | Dots: Outliers',
        transform=ax.transAxes, fontsize=9, va='top', color='gray')

plt.tight_layout()
plt.savefig('plots/07_duration_by_rating.png', bbox_inches='tight')
plt.show()

---
## 6. Time Series — How Netflix Content Grew Over Time <a id='6'></a>

In [ ]:
# ── Safety: re-import if kernel was restarted mid-notebook ───────────────
try:
    sns, plt, pd, NETFLIX_RED
except NameError:
    import pandas as pd, seaborn as sns, matplotlib.pyplot as plt
    from wordcloud import WordCloud, STOPWORDS
    import os, warnings; warnings.filterwarnings('ignore')
    os.makedirs('plots', exist_ok=True)
    sns.set_theme(style='darkgrid')
    plt.rcParams.update({'figure.dpi':120,'axes.titlesize':15,'axes.labelsize':12})
    NETFLIX_RED='#E50914'; NETFLIX_DARK='#141414'
    print('⚠️  Re-imported libraries. Run Kernel → Restart & Run All for best results.')

# ─────────────────────────────────────────────────────────────────────────────
# 6.1  Total Titles Added Per Year
# ─────────────────────────────────────────────────────────────────────────────
yearly = df.groupby(['year_added', 'type']).size().unstack(fill_value=0)
# .groupby groups rows by year + type, .size() counts them
# .unstack() pivots 'type' column into separate columns

fig, ax = plt.subplots(figsize=(14, 6))
yearly.plot(kind='area', ax=ax, alpha=0.7,
            color=[NETFLIX_RED, '#555'], stacked=True)
ax.set_title('Netflix Content Added Per Year', fontsize=16)
ax.set_xlabel('Year')
ax.set_ylabel('Titles Added')
ax.legend(title='Type', loc='upper left')

# Annotate the peak year
peak_year = yearly.sum(axis=1).idxmax()
peak_val  = yearly.sum(axis=1).max()
ax.annotate(f'Peak: {peak_year}\n({peak_val} titles)',
            xy=(peak_year, peak_val),
            xytext=(peak_year - 2, peak_val - 200),
            arrowprops=dict(arrowstyle='->', color='white'),
            color='white', fontsize=10,
            bbox=dict(boxstyle='round,pad=0.3', fc=NETFLIX_RED, alpha=0.8))

plt.tight_layout()
plt.savefig('plots/08_yearly_growth.png', bbox_inches='tight')
plt.show()

print(f'\n🎯 Netflix added the most content in {peak_year}: {peak_val} titles!')

In [ ]:
# ── Safety: re-import if kernel was restarted mid-notebook ───────────────
try:
    sns, plt, pd, NETFLIX_RED
except NameError:
    import pandas as pd, seaborn as sns, matplotlib.pyplot as plt
    from wordcloud import WordCloud, STOPWORDS
    import os, warnings; warnings.filterwarnings('ignore')
    os.makedirs('plots', exist_ok=True)
    sns.set_theme(style='darkgrid')
    plt.rcParams.update({'figure.dpi':120,'axes.titlesize':15,'axes.labelsize':12})
    NETFLIX_RED='#E50914'; NETFLIX_DARK='#141414'
    print('⚠️  Re-imported libraries. Run Kernel → Restart & Run All for best results.')

# ─────────────────────────────────────────────────────────────────────────────
# 6.2  Which Month Does Netflix Add the Most Content?
# ─────────────────────────────────────────────────────────────────────────────
month_order = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
monthly = df.groupby('month_name').size().reindex(month_order)

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(monthly.index, monthly.values,
              color=[NETFLIX_RED if v == monthly.max() else '#555' for v in monthly.values],
              edgecolor='white')
ax.set_title('Content Added by Month (all years combined)')
ax.set_xlabel('Month')
ax.set_ylabel('Number of Titles')

# Highlight the peak month
best_month = monthly.idxmax()
ax.text(0.5, 0.95, f'📅 Most additions in: {best_month}',
        transform=ax.transAxes, ha='center', fontsize=11,
        color=NETFLIX_RED, fontweight='bold')

plt.tight_layout()
plt.savefig('plots/09_monthly_additions.png', bbox_inches='tight')
plt.show()

---
## 7. Genre Deep Dive <a id='7'></a>

Each title can have **multiple genres** separated by commas. We'll split them and count each one individually.

In [ ]:
# ── Safety: re-import if kernel was restarted mid-notebook ───────────────
try:
    sns, plt, pd, NETFLIX_RED
except NameError:
    import pandas as pd, seaborn as sns, matplotlib.pyplot as plt
    from wordcloud import WordCloud, STOPWORDS
    import os, warnings; warnings.filterwarnings('ignore')
    os.makedirs('plots', exist_ok=True)
    sns.set_theme(style='darkgrid')
    plt.rcParams.update({'figure.dpi':120,'axes.titlesize':15,'axes.labelsize':12})
    NETFLIX_RED='#E50914'; NETFLIX_DARK='#141414'
    print('⚠️  Re-imported libraries. Run Kernel → Restart & Run All for best results.')

# ─────────────────────────────────────────────────────────────────────────────
# 7.1  Explode multi-genre column into individual rows
# ─────────────────────────────────────────────────────────────────────────────
# str.split() turns 'Drama, Comedy' into ['Drama', 'Comedy']
# explode() turns each list item into its own row
genre_series = df['listed_in'].str.split(', ').explode().str.strip()
genre_counts = genre_series.value_counts().head(20)

fig, ax = plt.subplots(figsize=(12, 8))
# Horizontal bar — good for long category names
sns.barplot(x=genre_counts.values, y=genre_counts.index,
            palette='Reds_r', ax=ax)
ax.set_title('Top 20 Genres on Netflix', fontsize=16)
ax.set_xlabel('Number of Titles')
ax.set_ylabel('')

# Add value labels
for i, (val, name) in enumerate(zip(genre_counts.values, genre_counts.index)):
    ax.text(val + 10, i, str(val), va='center', fontsize=9)

plt.tight_layout()
plt.savefig('plots/10_top_genres.png', bbox_inches='tight')
plt.show()

---
## 8. Country-wise Analysis <a id='8'></a>

In [ ]:
# ── Safety: re-import if kernel was restarted mid-notebook ───────────────
try:
    sns, plt, pd, NETFLIX_RED
except NameError:
    import pandas as pd, seaborn as sns, matplotlib.pyplot as plt
    from wordcloud import WordCloud, STOPWORDS
    import os, warnings; warnings.filterwarnings('ignore')
    os.makedirs('plots', exist_ok=True)
    sns.set_theme(style='darkgrid')
    plt.rcParams.update({'figure.dpi':120,'axes.titlesize':15,'axes.labelsize':12})
    NETFLIX_RED='#E50914'; NETFLIX_DARK='#141414'
    print('⚠️  Re-imported libraries. Run Kernel → Restart & Run All for best results.')

# ─────────────────────────────────────────────────────────────────────────────
# 8.1  Top 15 Countries by Number of Titles
# ─────────────────────────────────────────────────────────────────────────────
# Some titles have multiple countries — we explode similarly to genres
country_series = df[df['country'] != 'Unknown']['country'].str.split(', ').explode().str.strip()
top_countries = country_series.value_counts().head(15)

fig, ax = plt.subplots(figsize=(12, 7))
colors = [NETFLIX_RED if c == 'United States' else '#555' for c in top_countries.index]
sns.barplot(x=top_countries.values, y=top_countries.index, palette=None, ax=ax)
ax.patches  # seaborn won't accept palette=None so:

ax.barh(top_countries.index[::-1], top_countries.values[::-1], color=colors[::-1])
ax.set_title('Top 15 Content-Producing Countries on Netflix', fontsize=14)
ax.set_xlabel('Number of Titles')
plt.tight_layout()
plt.savefig('plots/11_top_countries.png', bbox_inches='tight')
plt.show()

print(f'\n🌍 US dominates with {top_countries["United States"]:,} titles — \
{top_countries["United States"]/country_series.shape[0]*100:.1f}% of all country-tagged content.')

In [ ]:
# ── Safety: re-import if kernel was restarted mid-notebook ───────────────
try:
    sns, plt, pd, NETFLIX_RED
except NameError:
    import pandas as pd, seaborn as sns, matplotlib.pyplot as plt
    from wordcloud import WordCloud, STOPWORDS
    import os, warnings; warnings.filterwarnings('ignore')
    os.makedirs('plots', exist_ok=True)
    sns.set_theme(style='darkgrid')
    plt.rcParams.update({'figure.dpi':120,'axes.titlesize':15,'axes.labelsize':12})
    NETFLIX_RED='#E50914'; NETFLIX_DARK='#141414'
    print('⚠️  Re-imported libraries. Run Kernel → Restart & Run All for best results.')

# ─────────────────────────────────────────────────────────────────────────────
# 8.2  India focus — Movies vs TV Shows
# ─────────────────────────────────────────────────────────────────────────────
india_df = df[df['country'].str.contains('India', na=False)]
india_type = india_df['type'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('India Content on Netflix', fontsize=15, fontweight='bold')

axes[0].pie(india_type, labels=india_type.index, autopct='%1.1f%%',
            colors=[NETFLIX_RED, '#333'], startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[0].set_title('Movies vs TV Shows')

india_genre = india_df['listed_in'].str.split(', ').explode().str.strip().value_counts().head(10)
sns.barplot(x=india_genre.values, y=india_genre.index, palette='Reds_r', ax=axes[1])
axes[1].set_title('Top Genres in Indian Content')
axes[1].set_xlabel('Count')

plt.tight_layout()
plt.savefig('plots/12_india_content.png', bbox_inches='tight')
plt.show()

print(f'\n🇮🇳 India has {len(india_df):,} titles on Netflix!')

---
## 9. Word Cloud — What Are Netflix Descriptions About? <a id='9'></a>

A **Word Cloud** shows words in sizes proportional to how often they appear. Bigger = more frequent.

In [ ]:
# ── Safety: re-import if kernel was restarted mid-notebook ───────────────
try:
    sns, plt, pd, NETFLIX_RED
except NameError:
    import pandas as pd, seaborn as sns, matplotlib.pyplot as plt
    from wordcloud import WordCloud, STOPWORDS
    import os, warnings; warnings.filterwarnings('ignore')
    os.makedirs('plots', exist_ok=True)
    sns.set_theme(style='darkgrid')
    plt.rcParams.update({'figure.dpi':120,'axes.titlesize':15,'axes.labelsize':12})
    NETFLIX_RED='#E50914'; NETFLIX_DARK='#141414'
    print('⚠️  Re-imported libraries. Run Kernel → Restart & Run All for best results.')

from wordcloud import WordCloud, STOPWORDS

# Join all descriptions into one big text string
all_text = ' '.join(df['description'].dropna().tolist())

# Stopwords = common words to IGNORE (the, a, is, etc.)
stopwords = set(STOPWORDS)
stopwords.update(['one', 'two', 'find', 'life', 'new', 'will', 'must', 'world'])

wc = WordCloud(
    width=1200, height=600,
    background_color='black',
    colormap='Reds',
    stopwords=stopwords,
    max_words=150,
    collocations=False   # don't pair words, treat them individually
).generate(all_text)

fig, ax = plt.subplots(figsize=(16, 8))
ax.imshow(wc, interpolation='bilinear')
ax.axis('off')  # hide the x/y axes
ax.set_title('Word Cloud — Netflix Show & Movie Descriptions', fontsize=18, pad=12)
plt.tight_layout()
plt.savefig('plots/13_wordcloud.png', bbox_inches='tight', facecolor='black')
plt.show()
print('☁️ Word cloud generated!')

---
## 10. Key Insights & Conclusions <a id='10'></a>

In [ ]:
# ── Safety: re-import if kernel was restarted mid-notebook ───────────────
try:
    sns, plt, pd, NETFLIX_RED
except NameError:
    import pandas as pd, seaborn as sns, matplotlib.pyplot as plt
    from wordcloud import WordCloud, STOPWORDS
    import os, warnings; warnings.filterwarnings('ignore')
    os.makedirs('plots', exist_ok=True)
    sns.set_theme(style='darkgrid')
    plt.rcParams.update({'figure.dpi':120,'axes.titlesize':15,'axes.labelsize':12})
    NETFLIX_RED='#E50914'; NETFLIX_DARK='#141414'
    print('⚠️  Re-imported libraries. Run Kernel → Restart & Run All for best results.')

# ─────────────────────────────────────────────────────────────────────────────
# 10.1  Final Summary Dashboard
# ─────────────────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(16, 10))
fig.patch.set_facecolor(NETFLIX_DARK)
fig.suptitle('Netflix EDA — Key Metrics Dashboard',
             fontsize=20, color='white', fontweight='bold', y=0.98)

metrics = [
    ('Total Titles', f'{len(df):,}', '🎬'),
    ('Movies', f'{(df["type"]=="Movie").sum():,}', '🎥'),
    ('TV Shows', f'{(df["type"]=="TV Show").sum():,}', '📺'),
    ('Countries', f'{country_series.nunique()}', '🌍'),
    ('Avg Movie Duration', f'{movies["duration_value"].mean():.0f} min', '⏱️'),
    ('Year Range', f'{df["release_year"].min()}–{df["release_year"].max()}', '📅'),
]

for i, (label, value, icon) in enumerate(metrics):
    ax = fig.add_subplot(2, 3, i + 1)
    ax.set_facecolor('#1a1a1a')
    ax.text(0.5, 0.6, f'{icon}\n{value}', ha='center', va='center',
            fontsize=22, color=NETFLIX_RED, fontweight='bold',
            transform=ax.transAxes, linespacing=1.4)
    ax.text(0.5, 0.18, label, ha='center', va='bottom',
            fontsize=12, color='white', transform=ax.transAxes)
    ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_edgecolor('#444')

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig('plots/14_dashboard.png', bbox_inches='tight', facecolor=NETFLIX_DARK)
plt.show()

---

## ✅ Summary of Key Insights

| # | Insight |
|---|---|
| 1 | **Movies dominate** Netflix — about 70% of all content |
| 2 | **TV-MA** is the most common rating — Netflix skews toward mature audiences |
| 3 | The **average movie** on Netflix is ~99 minutes long |
| 4 | **Most TV shows** have only 1 season — Netflix cancels many early |
| 5 | Content additions **peaked in 2019–2020**, then declined (COVID / market saturation) |
| 6 | **July & December** are the biggest months for new additions |
| 7 | **International Movies & Dramas** are the top genres globally |
| 8 | **USA leads** content production; India is #2 — 🇮🇳 Bollywood is big on Netflix! |
| 9 | Description keywords: *family, love, young, story* — emotional storytelling dominates |

---

**🔗 Dataset Source:** [Netflix Movies and TV Shows — Kaggle](https://www.kaggle.com/datasets/shivamb/netflix-shows)  
**👤 Author:** Your Name  
**📅 Date:** 2024
